# Bidirectional LSTM

The network is trained on the binary training file. Epochs are chosen with a
validation slice cut from that file. Metrics are computed on the held-out test
file, which is not passed to `fit` and is not the early-stopping signal.

`NUM_RUNS` defaults to 30 and only changes the initialisation seed. The
train/validation/test prompts stay fixed and disjoint.

In [ ]:
import os
import sys
from pathlib import Path

HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / "code").is_dir() else HERE.parent
sys.path.insert(0, str(ROOT / "code"))

import numpy as np
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Bidirectional, Dense, Embedding, LSTM, SpatialDropout1D
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

from binary_data import load_binary_frames
from latex_tables import export_results
from metrics import classification_metrics
from splits import split_train_val_test

NUM_RUNS = int(os.environ.get("NUM_RUNS", "30"))
train_df, test_df = load_binary_frames()
train_part, val_part, test_part, report = split_train_val_test(
    train_df, test_df, val_fraction=0.2, random_state=0
)
print(
    f"train={len(train_part)} val={len(val_part)} test={len(test_part)} "
    f"shared_removed={report['shared_prompts_before']}"
)

max_words = 5000
max_len = 100
tokenizer = Tokenizer(num_words=max_words, lower=True, split=" ")
tokenizer.fit_on_texts(train_part["prompt"])
x_train = pad_sequences(tokenizer.texts_to_sequences(train_part["prompt"]), maxlen=max_len)
x_val = pad_sequences(tokenizer.texts_to_sequences(val_part["prompt"]), maxlen=max_len)
x_test = pad_sequences(tokenizer.texts_to_sequences(test_part["prompt"]), maxlen=max_len)
y_train = train_part["jailbreak"].astype(int).to_numpy()
y_val = val_part["jailbreak"].astype(int).to_numpy()
y_test = test_part["jailbreak"].astype(int).to_numpy()

runs = []
for run in range(NUM_RUNS):
    tf.random.set_seed(run)
    model = Sequential()
    model.add(Embedding(max_words, 100))
    model.add(SpatialDropout1D(0.2))
    model.add(Bidirectional(LSTM(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.2)))
    model.add(Bidirectional(LSTM(64, dropout=0.2, recurrent_dropout=0.2)))
    model.add(Dense(128, activation="relu"))
    model.add(Dense(64, activation="relu"))
    model.add(Dense(32, activation="relu"))
    model.add(Dense(1, activation="sigmoid"))
    model.compile(
        loss="binary_crossentropy",
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        metrics=["accuracy"],
    )
    model.fit(
        x_train,
        y_train,
        epochs=40,
        batch_size=32,
        validation_data=(x_val, y_val),
        callbacks=[EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)],
        verbose=0,
    )
    scores = model.predict(x_test, verbose=0).ravel()
    predictions = (scores >= 0.5).astype(int)
    metrics = classification_metrics(y_test, predictions, scores)
    runs.append(
        {
            "features": "Tokens",
            "model": "BD-LSTM",
            "auc": metrics["AUC"],
            "accuracy": metrics["Accuracy"],
            "fnr": metrics["FNR"],
            "tpr": metrics["TPR"],
            "run": run,
        }
    )
    print(f"run {run + 1}/{NUM_RUNS} AUC={metrics['AUC']:.4f} accuracy={metrics['Accuracy']:.4f}")

paths = export_results("03_bilstm", "binary", runs)
print(f"Wrote {paths['tex']}")